<a href="https://colab.research.google.com/github/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_For_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC for Colab

基于 [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI)，适配 Colab 的 Python 3.11 环境。
Based on [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI), adapted for the Python 3.11 runtime on Colab.

1. 运行前请选择 GPU 运行时：`代码执行程序 > 更改运行时类型 > T4 GPU`。
   Select a GPU runtime first: `Runtime > Change runtime type > T4 GPU`.
2. 按顺序运行“准备环境”中的单元格，然后选择 **WebUI** 或 **命令行训练** 其中一种方式。
   Run the cells under "Setup" in order, then use either the **WebUI** or the **command-line training** section.

项目主页 / Project page: https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab

# 准备环境 / Setup

In [ ]:
# @title 克隆仓库 / Clone the repository
%cd /content
!rm -rf /content/RVC-Retrieval-based-Voice-Conversion-Colab
!git clone --depth=1 https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab.git
%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab

In [ ]:
# @title 安装依赖 / Install dependencies
%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
!pip install -r requirements.txt

In [ ]:
# @title 下载预训练模型 / Download pretrained models
# @markdown 下载 v1 底模 / Also download v1 pretrained models
DOWNLOAD_V1 = False  # @param {type:"boolean"}
# @markdown 下载 UVR5 人声分离模型 / Download UVR5 vocal separation models
DOWNLOAD_UVR5 = True  # @param {type:"boolean"}

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
!apt-get -qq install -y aria2 > /dev/null

HF = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
names = [f"{f0}{gd}{sr}.pth" for f0 in ("", "f0") for gd in ("G", "D") for sr in ("32k", "40k", "48k")]
files = [(f"{HF}/pretrained_v2/{n}", "assets/pretrained_v2", n) for n in names]
if DOWNLOAD_V1:
    files += [(f"{HF}/pretrained/{n}", "assets/pretrained", n) for n in names]
if DOWNLOAD_UVR5:
    for n in ("HP2_all_vocals.pth", "HP3_all_vocals.pth", "HP5_only_main_vocal.pth",
              "VR-DeEchoAggressive.pth", "VR-DeEchoDeReverb.pth", "VR-DeEchoNormal.pth"):
        files.append((f"{HF}/uvr5_weights/{n}", "assets/uvr5_weights", n))
files += [
    (f"{HF}/hubert_base.pt", "assets/hubert", "hubert_base.pt"),
    (f"{HF}/rmvpe.pt", "assets/rmvpe", "rmvpe.pt"),
]

with open("/tmp/rvc_models.txt", "w") as f:
    for url, out_dir, name in files:
        f.write(f"{url}\n  dir={out_dir}\n  out={name}\n")
!aria2c --console-log-level=error --summary-interval=0 -c -x 16 -s 16 -k 1M -j 4 -i /tmp/rvc_models.txt

In [ ]:
# @title 挂载谷歌云盘 / Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 从谷歌云盘解压数据集到 /content/dataset / Unzip dataset from Google Drive to /content/dataset
# @markdown 数据集 zip 路径（zip 内直接放音频文件） / Path of the dataset zip (audio files at the top level)
DATASET = "/content/drive/MyDrive/datasets/RVC/src.zip"  # @param {type:"string"}

!mkdir -p /content/dataset
!unzip -o -q -d /content/dataset "{DATASET}"
!ls /content/dataset | head

In [ ]:
# @title 从谷歌云盘恢复模型（首次使用无需执行） / Restore a model from Google Drive (skip on first use)
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要恢复的 epoch；“仅保存最新”时填 2333333 / Epoch to restore; use 2333333 if "only save latest" was on
MODELEPOCH = 200  # @param {type:"integer"}
# @markdown 备份所在目录 / Backup folder on Drive
BACKUP_DIR = "/content/drive/MyDrive/RVC_backup"  # @param {type:"string"}
# @markdown 旧版笔记本的备份（文件直接放在 MyDrive 根目录，且 G/D 文件名互换） / Backup made by the old notebook (files in MyDrive root, G/D names swapped)
LEGACY_BACKUP = False  # @param {type:"boolean"}

import glob, os, shutil

os.chdir("/content/RVC-Retrieval-based-Voice-Conversion-Colab")
exp_dir = f"logs/{MODELNAME}"
os.makedirs(exp_dir, exist_ok=True)
os.makedirs("assets/weights", exist_ok=True)

if LEGACY_BACKUP:
    src = "/content/drive/MyDrive"
    shutil.copy(f"{src}/{MODELNAME}_D_{MODELEPOCH}.pth", f"{exp_dir}/G_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}_G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}{MODELEPOCH}.pth", f"assets/weights/{MODELNAME}.pth")
    extra = glob.glob(f"{src}/added_*{MODELNAME}*.index") + glob.glob(f"{src}/total_*.npy")
else:
    src = f"{BACKUP_DIR}/{MODELNAME}"
    for gd in ("G", "D"):
        shutil.copy(f"{src}/{gd}_{MODELEPOCH}.pth", exp_dir)
    for f in glob.glob(f"{src}/{MODELNAME}*.pth"):
        shutil.copy(f, "assets/weights")
    extra = glob.glob(f"{src}/*.index") + glob.glob(f"{src}/*.npy") + glob.glob(f"{src}/config.json")
for f in extra:
    shutil.copy(f, exp_dir)
print("Restored to", exp_dir, sorted(os.listdir(exp_dir)))

# 方式一：WebUI / Option 1: WebUI

In [ ]:
# @title 方案一：Gradio 公网链接 / Plan A: Gradio share link
%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
#%load_ext tensorboard
#%tensorboard --logdir /content/RVC-Retrieval-based-Voice-Conversion-Colab/logs
!python infer-web.py --colab --pycmd python3 --port 7890

In [ ]:
# @title 方案二：ngrok 隧道 / Plan B: ngrok tunnel
# @markdown 在 https://dashboard.ngrok.com 获取 Authtoken / Get your authtoken at https://dashboard.ngrok.com
Authtoken = ""  # @param {type:"string"}

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
!pip -q install pyngrok
from pyngrok import ngrok

if not Authtoken:
    raise ValueError("请先填写 ngrok Authtoken / Please fill in your ngrok authtoken")
ngrok.set_auth_token(Authtoken)
public_url = ngrok.connect(7860, "http")
print("ngrok 公网访问地址 / Public URL:", public_url.public_url)
!python infer-web.py --pycmd python3 --port 7860 --noautoopen

# 方式二：命令行训练 / Option 2: command-line training
若运行 WebUI 时提示该代码被禁止执行，可以用下面的单元格直接训练。
If Colab refuses to run the WebUI, train with the cells below instead.

所有单元格中的模型名、采样率和版本需保持一致。
Keep the model name, sample rate and version the same in every cell.

In [ ]:
# @title 音频预处理 / Preprocess audio
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["32k", "40k", "48k"]
# @markdown 使用的进程数 / Number of processes
THREADCOUNT = 8  # @param {type:"integer"}

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
SR_HZ = int(SAMPLERATE[:-1]) * 1000
!mkdir -p logs/{MODELNAME}
!python infer/modules/train/preprocess.py /content/dataset {SR_HZ} {THREADCOUNT} logs/{MODELNAME} False 3.7

In [ ]:
# @title 提取音高与特征 / Extract pitch and features
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 音高提取算法，rmvpe_gpu 最快且效果最好 / Pitch extraction method, rmvpe_gpu is the fastest and usually the best
ALGO = "rmvpe_gpu"  # @param ["rmvpe_gpu", "rmvpe", "harvest", "pm", "dio"]
# @markdown 使用的进程数（rmvpe_gpu 不使用） / Number of processes (not used by rmvpe_gpu)
THREADCOUNT = 8  # @param {type:"integer"}

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
if ALGO == "rmvpe_gpu":
    !python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 logs/{MODELNAME} True
else:
    !python infer/modules/train/extract/extract_f0_print.py logs/{MODELNAME} {THREADCOUNT} {ALGO}
!python infer/modules/train/extract_feature_print.py cuda 1 0 0 logs/{MODELNAME} {VERSION} True

In [ ]:
# @title 训练模型 / Train the model
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["32k", "40k", "48k"]
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 批大小 / Batch size
BATCHSIZE = 8  # @param {type:"integer"}
# @markdown 总 epoch 数 / Total epochs
MODELEPOCH = 200  # @param {type:"integer"}
# @markdown 保存间隔（epoch） / Save every N epochs
EPOCHSAVE = 10  # @param {type:"integer"}
# @markdown 是否仅保存最新的 G/D 检查点（文件名为 G_2333333.pth） / Only keep the latest G/D checkpoint (saved as G_2333333.pth)
ONLYLATEST = True  # @param {type:"boolean"}
# @markdown 是否把训练集缓存到显存（数据集较小时可加速） / Cache the dataset in GPU memory (faster for small datasets)
CACHEDATA = False  # @param {type:"boolean"}
# @markdown 每次保存时都导出一个小模型到 assets/weights / Export a small model to assets/weights at every save
SAVEWEIGHTS = True  # @param {type:"boolean"}
# @markdown 在下方显示 TensorBoard / Show TensorBoard below
TENSORBOARD = True  # @param {type:"boolean"}

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
if TENSORBOARD:
    %load_ext tensorboard
    %tensorboard --logdir logs
PRETRAINED = "assets/pretrained_v2" if VERSION == "v2" else "assets/pretrained"
!python tools/colab_train.py filelist {MODELNAME} {SAMPLERATE} {VERSION}
!python infer/modules/train/train.py -e {MODELNAME} -sr {SAMPLERATE} -f0 1 -bs {BATCHSIZE} -g 0 -te {MODELEPOCH} -se {EPOCHSAVE} -pg {PRETRAINED}/f0G{SAMPLERATE}.pth -pd {PRETRAINED}/f0D{SAMPLERATE}.pth -l {int(ONLYLATEST)} -c {int(CACHEDATA)} -sw {int(SAVEWEIGHTS)} -v {VERSION}

In [ ]:
# @title 训练特征索引 / Train the feature index
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]

%cd /content/RVC-Retrieval-based-Voice-Conversion-Colab
!python tools/colab_train.py index {MODELNAME} {VERSION}

# 备份 / Backup

In [ ]:
# @title 备份模型到谷歌云盘 / Back up the model to Google Drive
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要备份的 epoch；“仅保存最新”时填 2333333 / Epoch to back up; use 2333333 if "only save latest" was on
MODELEPOCH = 200  # @param {type:"integer"}
# @markdown 备份目录 / Backup folder on Drive
BACKUP_DIR = "/content/drive/MyDrive/RVC_backup"  # @param {type:"string"}

import glob, os, shutil

os.chdir("/content/RVC-Retrieval-based-Voice-Conversion-Colab")
exp_dir = f"logs/{MODELNAME}"
dst = f"{BACKUP_DIR}/{MODELNAME}"
os.makedirs(dst, exist_ok=True)
files = [f"{exp_dir}/G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth", f"{exp_dir}/config.json"]
files += glob.glob(f"{exp_dir}/added_*.index") + glob.glob(f"{exp_dir}/total_*.npy")
files += glob.glob(f"assets/weights/{MODELNAME}*.pth")
for f in files:
    if os.path.exists(f):
        shutil.copy(f, dst)
        print("copied", f)
    else:
        print("missing", f)
print("Backup folder:", dst)